# 02 — Análisis de Aforos de Tráfico de Madrid

Exploración del dataset de aforos históricos de Open Data Madrid.
Objetivo: entender los patrones de congestión por hora, día y distrito
antes de entrenar el modelo de predicción.

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ml.generate_dataset import generate_and_save, DISTRITOS_MADRID

sns.set_theme(style='darkgrid')
plt.rcParams['figure.dpi'] = 110

## 1. Carga del dataset

Si los aforos reales están disponibles en `data/raw/open_data_madrid/aforos_historicos.csv`,
se cargan directamente. De lo contrario se genera el dataset sintético.

In [ ]:
AFOROS_REAL = '../data/raw/open_data_madrid/aforos_historicos.csv'
DATASET_PATH = '../data/processed/aforos_dataset.csv'

if not os.path.exists(DATASET_PATH):
    print('Generando dataset...')
    generate_and_save()

df = pd.read_csv(DATASET_PATH)
print(f'Filas: {len(df):,}  |  Columnas: {list(df.columns)}')
df.head()

## 2. Distribución del nivel de tráfico

In [ ]:
label_map = {0: 'Bajo', 1: 'Medio', 2: 'Alto'}
df['nivel_label'] = df['nivel_trafico'].map(label_map)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['nivel_trafico'].value_counts().sort_index()
counts.index = counts.index.map(label_map)
counts.plot(kind='bar', ax=axes[0], color=['#22c55e', '#f59e0b', '#ef4444'], edgecolor='white', rot=0)
axes[0].set_title('Distribución global de niveles de tráfico')
axes[0].set_ylabel('Número de registros')

df['nivel_trafico'].value_counts(normalize=True).sort_index().mul(100).plot(
    kind='pie', ax=axes[1], labels=['Bajo', 'Medio', 'Alto'],
    colors=['#22c55e', '#f59e0b', '#ef4444'], autopct='%1.1f%%', startangle=90
)
axes[1].set_title('Proporción de cada nivel')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 3. Tráfico por hora del día

In [ ]:
pivot_hora = df.groupby('hora')['nivel_trafico'].mean()

fig, ax = plt.subplots(figsize=(11, 4))
ax.fill_between(pivot_hora.index, pivot_hora.values, alpha=0.3, color='steelblue')
ax.plot(pivot_hora.index, pivot_hora.values, color='steelblue', linewidth=2, marker='o', markersize=4)
ax.axhspan(0, 0.5, alpha=0.07, color='green', label='Bajo')
ax.axhspan(0.5, 1.5, alpha=0.07, color='orange', label='Medio')
ax.axhspan(1.5, 2.0, alpha=0.07, color='red', label='Alto')
ax.set_xticks(range(0, 24))
ax.set_title('Nivel de tráfico medio por hora del día')
ax.set_xlabel('Hora')
ax.set_ylabel('Nivel medio (0=Bajo, 2=Alto)')
ax.legend()
plt.tight_layout()
plt.show()

print('Horas punta (nivel > 1.0):')
print(pivot_hora[pivot_hora > 1.0].to_string())

## 4. Tráfico por día de la semana

In [ ]:
dias = ['Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado', 'Domingo']
pivot_dia = df.groupby('dia_semana')['nivel_trafico'].mean()
pivot_dia.index = [dias[i] for i in pivot_dia.index]

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#ef4444' if d in ['Sábado', 'Domingo'] else 'steelblue' for d in pivot_dia.index]
pivot_dia.plot(kind='bar', ax=ax, color=colors, edgecolor='white', rot=30)
ax.set_title('Nivel de tráfico medio por día de la semana')
ax.set_ylabel('Nivel medio')
plt.tight_layout()
plt.show()

## 5. Heatmap hora × día

In [ ]:
heatmap_data = df.groupby(['dia_semana', 'hora'])['nivel_trafico'].mean().unstack()
heatmap_data.index = dias

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    heatmap_data, ax=ax, cmap='RdYlGn_r',
    linewidths=0.5, linecolor='#1e293b',
    cbar_kws={'label': 'Nivel tráfico (0=Bajo, 2=Alto)'}
)
ax.set_title('Heatmap de tráfico — Hora × Día de la semana', fontsize=13)
ax.set_xlabel('Hora del día')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 6. Tráfico por distrito

In [ ]:
pivot_distrito = df.groupby('distrito')['nivel_trafico'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 7))
pivot_distrito.plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Nivel de tráfico medio por distrito de Madrid')
ax.set_xlabel('Nivel medio (0=Bajo, 2=Alto)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 7. Impacto de festivos vs. laborables

In [ ]:
comparison = df.groupby(['es_fin_de_semana', 'es_festivo'])['nivel_trafico'].mean().reset_index()
comparison['tipo'] = comparison.apply(
    lambda r: 'Festivo' if r['es_festivo'] == 1
    else ('Fin de semana' if r['es_fin_de_semana'] == 1 else 'Laborable'),
    axis=1
)
print(comparison[['tipo', 'nivel_trafico']].to_string(index=False))